# Thesis experiments — complete run on Google Colab

Everything for Chapters 4–5 runs here, on **one Colab T4 GPU**: all 30 trainings
(5 models × 2 categories × 3 seeds), Exp 1–3 (accuracy, degradation, lighting),
Exp 4 (latency), Exp 5 (1 image/second stream), and all figures and tables.

## Before you start
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**, and allow Google Drive access (the account holding `thesis_project_ASUS.zip`)
3. Keep this tab open, and look at it every hour or so so Colab does not mark it idle

## Expected time: about 3.5–4 hours
| Part | Time |
|---|---|
| Setup | ~10 min |
| Training, 30 models | ~1.5 h |
| Exp 1, 2+3 | ~30 min |
| Exp 4 latency (both categories) | ~20 min |
| Exp 5 stream (sheet_metal, 5 models × 500 images × 2 repeats at 1 image/s) | ~1.5 h |

Times printed below are Indian Standard Time.

## If Colab disconnects
**Runtime → Run all** again. Progress lives in Drive (`MyDrive/thesis_colab_outputs/`), so finished
steps are skipped. All timing must come from one GPU type: if a reconnect gives a different GPU,
the timing steps are automatically re-run on it.

## Result
`MyDrive/thesis_colab_results.zip` — models, results, figures, logs. Download it and give it to Claude.

In [ ]:
# 1. GPU check — must print a Tesla T4 (or another NVIDIA GPU)
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all again'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

In [ ]:
# 2. Connect Google Drive and use Indian time in the logs
import os, time
os.environ['TZ'] = 'Asia/Kolkata'; time.tzset()
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Find the project zip in Drive and unpack it to the fast local disk
import glob, shutil, subprocess
DRIVE = '/content/drive/MyDrive'
PROJECT = '/content/predictive-maintenance-cv'
hits = glob.glob(f'{DRIVE}/thesis_project_ASUS.zip') or glob.glob(f'{DRIVE}/**/thesis_project_ASUS.zip', recursive=True)
assert hits, 'thesis_project_ASUS.zip not found in Google Drive - upload it to My Drive'
if not os.path.isdir(PROJECT):
    print('unpacking', hits[0], '...')
    shutil.copy(hits[0], '/content/project.zip')      # unzipping straight from Drive is slow
    subprocess.run(['unzip', '-q', '/content/project.zip', '-d', '/content'], check=True)
    os.remove('/content/project.zip')
print('project ready:', PROJECT)

In [ ]:
# 4. Keep models, results, logs and figures in Drive so a disconnect loses nothing
OUT = f'{DRIVE}/thesis_colab_outputs'
os.makedirs(OUT, exist_ok=True)
for d in ('artifacts', 'results', 'logs', 'figures'):
    local, remote = f'{PROJECT}/{d}', f'{OUT}/{d}'
    if os.path.islink(local):
        continue
    if not os.path.exists(remote):
        shutil.copytree(local, remote)
    shutil.rmtree(local)
    os.symlink(remote, local)
print('outputs are saved in', OUT)

In [ ]:
%%writefile /content/predictive-maintenance-cv/scripts/run_queue.py
"""Run the whole experimental matrix on one machine, one job at a time (any OS).

  python scripts/run_queue.py                 # everything still missing
  python scripts/run_queue.py --only train    # just the training stage
  python scripts/run_queue.py --dry-run       # list what would run

Stages, in order: train -> eval (Exp 1, Exp 2+3) -> timing (Exp 4, Exp 5) -> figures.
Timing runs last so nothing else competes for the machine. Each step logs to
logs/<step>.log; finished steps are listed in logs/done.txt and skipped on re-run,
and a model whose calibration.json exists is never retrained, so the queue can be
restarted after any interruption. Exp 6 (online service) is not part of the queue.

Keep the machine awake and plugged in: on Windows set the power plan to never
sleep while plugged in; on macOS run it under `caffeinate -dims`.
"""
from __future__ import annotations

import argparse
import subprocess
import sys
from datetime import datetime
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
CATEGORIES = ["sheet_metal", "can"]
MODELS = ["patchcore", "cae", "resnet50", "efficientnet_b0", "vit"]
SEEDS = [0, 1, 2]
# Micro-batch x accumulation = effective batch 16 in every profile (README, decision D15).
# "laptop" fits an 8 GB machine; "colab" uses a 16 GB GPU and two data-loading workers.
PROFILES = {
    "laptop": {
        "vit": ["--set", "train.batch_size=4", "--set", "train.accumulation_steps=4"],
        "resnet50": ["--set", "train.batch_size=8", "--set", "train.accumulation_steps=2"],
        "efficientnet_b0": ["--set", "train.batch_size=8", "--set", "train.accumulation_steps=2"],
    },
    "colab": {
        "vit": ["--set", "train.batch_size=8", "--set", "train.accumulation_steps=2"],
        "*": ["--set", "runtime.num_workers=2"],
    },
}
MEM_ARGS: dict[str, list[str]] = {}


def mem_args(model: str) -> list[str]:
    return [*MEM_ARGS.get(model, []), *MEM_ARGS.get("*", [])]


def now() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def build_steps(stream_categories: list[str]) -> list[tuple[str, str, list[str]]]:
    """(stage, name, argv) in execution order."""
    py = sys.executable
    steps = []
    for cat in CATEGORIES:
        for m in MODELS:
            for s in SEEDS:
                steps.append(("train", f"train_{m}_{cat}_seed{s}",
                              [py, "scripts/02_train.py", "--model", m, "--category", cat, "--seed", str(s),
                               *mem_args(m)]))
    for cat in CATEGORIES:
        for m in MODELS:
            steps.append(("eval", f"exp1_{m}_{cat}",
                          [py, "scripts/03_eval_nominal.py", "--model", m, "--category", cat, *mem_args(m)]))
            steps.append(("eval", f"exp2_{m}_{cat}",
                          [py, "scripts/04_eval_reliability.py", "--model", m, "--category", cat,
                           *mem_args(m)]))
    for cat in CATEGORIES:
        for m in MODELS:
            steps.append(("timing", f"exp4_{m}_{cat}", [py, "scripts/05_bench_latency.py", "--model", m, "--category", cat]))
            if cat in stream_categories:
                steps.append(("timing", f"exp5_{m}_{cat}",
                              [py, "scripts/06_stream.py", "--model", m, "--category", cat]))
    steps.append(("figures", "figures", [py, "scripts/08_make_figures.py"]))
    return steps


def already_trained(name: str) -> bool:
    # train_<model>_<category>_seed<s>; model and category names may contain underscores
    for m in MODELS:
        for cat in CATEGORIES:
            prefix = f"train_{m}_{cat}_seed"
            if name.startswith(prefix):
                seed = name[len(prefix):]
                return (ROOT / "artifacts" / cat / m / f"seed{seed}" / "calibration.json").exists()
    return False


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--only", nargs="+", choices=["train", "eval", "timing", "figures"],
                        help="run only these stages")
    parser.add_argument("--dry-run", action="store_true")
    parser.add_argument("--profile", choices=sorted(PROFILES), default="laptop",
                        help="batch/worker settings: laptop (8 GB) or colab (16 GB GPU)")
    parser.add_argument("--stream-categories", nargs="+", choices=CATEGORIES, default=CATEGORIES,
                        help="categories that get the Exp 5 stream (default: all)")
    args = parser.parse_args()
    MEM_ARGS.update(PROFILES[args.profile])

    logs = ROOT / "logs"
    logs.mkdir(exist_ok=True)
    done_file = logs / "done.txt"
    done = set(done_file.read_text().split()) if done_file.exists() else set()
    failed = []
    for stage, name, argv in build_steps(args.stream_categories):
        if args.only and stage not in args.only:
            continue
        if name in done or (stage == "train" and already_trained(name)):
            print(f"skip   {name}")
            continue
        if args.dry_run:
            print(f"would  {name}: {' '.join(argv[1:])}")
            continue
        print(f"[{now()}] start  {name}", flush=True)
        with open(logs / f"{name}.log", "w", encoding="utf-8") as fh:
            code = subprocess.run(argv, cwd=ROOT, stdout=fh, stderr=subprocess.STDOUT).returncode
        if code == 0:
            with open(done_file, "a", encoding="utf-8") as fh:
                fh.write(name + "\n")
            print(f"[{now()}] done   {name}", flush=True)
        else:
            failed.append(name)
            print(f"[{now()}] FAILED {name} (exit {code}, see logs/{name}.log)", flush=True)
    print("queue finished" + (f"; failed: {', '.join(failed)}" if failed else ""))


if __name__ == "__main__":
    main()


In [ ]:
# 6. Packages and project check (Colab already has PyTorch with CUDA)
!pip install -q timm psutil
%cd /content/predictive-maintenance-cv
!python scripts/check_setup.py

In [ ]:
# 7. Train everything on this GPU: remove any model that was not trained on CUDA
#    (the Mac-trained ones), so all 30 trainings happen here. Safe to re-run.
import json
removed = []
for meta in glob.glob(f'{PROJECT}/artifacts/*/*/seed*/train_meta.json'):
    if not json.load(open(meta)).get('device', '').startswith('cuda'):
        d = os.path.dirname(meta)
        cat, model, seed = d.split('/artifacts/')[1].split('/')
        shutil.rmtree(d)
        removed.append(f'train_{model}_{cat}_{seed}')
done_path = f'{PROJECT}/logs/done.txt'
if removed:   # their old evaluations are invalid too
    keep = [l for l in open(done_path).read().split()
            if l not in removed and not l.startswith(('exp', 'figures'))]
    open(done_path, 'w').write('\n'.join(keep) + '\n')
print(f'{len(removed)} non-Colab models removed' + (':' if removed else ' - all models are Colab-trained'))
print('\n'.join(sorted(removed)))

In [ ]:
# 8. Training — 30 models (~1.5 h). Each finished model is saved to Drive immediately.
!python scripts/run_queue.py --only train --profile colab

In [ ]:
# 9. Exp 1 (nominal accuracy) and Exp 2+3 (degradation, lighting) — ~30 min
!python scripts/run_queue.py --only eval --profile colab

In [ ]:
# 10. All timing must come from ONE GPU type. If this session's GPU differs from the one
#     that started the timing runs, clear them so every model is timed again on this GPU.
import os, torch
DRIVE = '/content/drive/MyDrive'
OUT = f'{DRIVE}/thesis_colab_outputs'
PROJECT = '/content/predictive-maintenance-cv'
done_path = f'{PROJECT}/logs/done.txt'
assert os.path.isdir(f'{PROJECT}/scripts') and os.path.islink(f'{PROJECT}/results'), \
    'This session is not set up (Colab restarted): use Runtime -> Run all'
gpu = torch.cuda.get_device_name(0)
marker = f'{OUT}/timing_gpu.txt'
if os.path.exists(marker) and open(marker).read().strip() != gpu:
    old = open(marker).read().strip()
    keep = [l for l in open(done_path).read().split() if not l.startswith(('exp4_', 'exp5_', 'figures'))]
    open(done_path, 'w').write('\n'.join(keep) + '\n')
    print(f'GPU changed ({old} -> {gpu}): all timing steps will be re-run')
open(marker, 'w').write(gpu)
print('timing GPU:', gpu)

In [ ]:
# 11. Exp 4 latency (both categories) and Exp 5 stream (sheet_metal) — ~2 h.
%cd /content/predictive-maintenance-cv
#     Do not run anything else in this Colab session while this cell runs.
!python scripts/run_queue.py --only timing --profile colab --stream-categories sheet_metal

In [ ]:
# 12. Figures and tables
%cd /content/predictive-maintenance-cv
!python scripts/run_queue.py --only figures
!ls figures/ results/

In [ ]:
# 13. Package everything -> MyDrive/thesis_colab_results.zip
%cd /content/predictive-maintenance-cv
!echo "finished steps: $(wc -l < logs/done.txt)"
!grep -l "Traceback" logs/*.log 2>/dev/null | head || true
!rm -f /content/thesis_colab_results.zip
!zip -q -r /content/thesis_colab_results.zip artifacts/ results/ logs/ figures/ -x '*.tmp'
!cp /content/thesis_colab_results.zip {DRIVE}/thesis_colab_results.zip
!ls -lh {DRIVE}/thesis_colab_results.zip
print('Done. Download MyDrive/thesis_colab_results.zip and give it to Claude.')